In [1]:
%matplotlib inline
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import os
import scipy.io as sio
import matplotlib
import logging

import utils as utils
from matplotlib import cm

In [2]:
runname = 'debug_withC_rogers_netcdfBdy1_EB_soliton_withsponge_focus_finey_summer'
datafilepath = '/Volumes/Tan_2024/SUNTANS/NLIW/iwave2024/dongsha/'+runname+'/data_nof_1wave_largeS/'

# datafilename = 'sun_out.nc_0000.nc'
datafilename = 'sun_sparse.nc_0000.nc'

# path to saved result
resultfilepath = datafilepath+'/results/'
# path to saved figures
outfilepath = '../figure/'
if not os.path.exists(resultfilepath):
    os.mkdir(resultfilepath)
if not os.path.exists(outfilepath):
    os.mkdir(outfilepath)

param_data = {
    "datafilepath": datafilepath,
    "netcdf_filename": datafilename,
            }
param_result = {
    "resultfilepath": resultfilepath,
    "result_filename": 'sun_out_energy_derived.nc',
            }
idx_end = -15

# Organize data output

In [13]:
# read data
Data = xr.open_dataset(datafilepath+datafilename)

# grid info|
import mat73
grid = mat73.loadmat('/Volumes/Tan_2024/SUNTANS/NLIW/iwave2024/dongsha/'+runname+'/mfiles/SUNTANS_grid.mat', only_include=['Nk', 'Nx', 'Ny', 'Depth']) 

Nx = grid['Nx'].astype('int')
Ny = grid['Ny'].astype('int')
Nk = grid['Nk'].astype('int')
Nc = Nx*Ny
# Depth = np.reshape(grid['Depth'].astype('int'), (Nx, Ny), order='F')
Depth = np.reshape(grid['Depth'], (Nx, Ny), order='F')

xv = np.reshape(Data.xv.data, (Nx, Ny), order='F')
yv = np.reshape(Data.yv.data, (Nx, Ny), order='F')

## _derived

In [ ]:
dEk_prime = np.reshape(Data.dEk_prime.data, (len(Data.time), Nx, Ny), order='F')
dEp_prime = np.reshape(Data.dEp_prime.data, (len(Data.time), Nx, Ny), order='F')

dEp0 = np.reshape(Data.dEp0.data, (len(Data.time), Nx, Ny), order='F')
dEk0 = np.reshape(Data.dEk0.data, (len(Data.time), Nx, Ny), order='F')

data = xr.Dataset({'dEp0': (['time','x', 'y'], dEp0),
                   'dEp_prime': (['time','x', 'y'], dEp_prime),
                   'dEk0': (['time','x', 'y'], dEk0),
                   'dEk_prime': (['time','x', 'y'], dEk_prime),
                  },
                  coords={'x': xv[:,0],
                          'y': yv[0,:],
                          'z': -Data.z_r.data,
                          'time': Data.time.data},
                  attrs={'title': 'SUNTANS internally calculated energy budget'}) 

del dEp0, dEp_prime, dEk0, dEk_prime

C1 = np.reshape(Data.C1_int.data, (len(Data.time), Nx, Ny), order='F')
C2 = np.reshape(Data.C2_int.data, (len(Data.time), Nx, Ny), order='F')
data["C_int"]=(['time', 'x', 'y'],  C1+C2)

del C1, C2

D_0 = np.reshape(Data.D_0_int.data, (len(Data.time), Nx, Ny), order='F')
D_prime = np.reshape(Data.D_prime_int.data, (len(Data.time), Nx, Ny), order='F')

data["D0"]=(['time', 'x', 'y'],  D_0)
data["D_prime"]=(['time', 'x', 'y'],  D_prime)

del D_0, D_prime

data["depth"]=(['x', 'y'],  Depth)

data.to_netcdf(param_result['resultfilepath']+param_result['result_filename'])

In [4]:
D_0 = np.reshape(Data.D_0_int.data, (len(Data.time), Nx, Ny), order='F')
D_prime = np.reshape(Data.D_prime_int.data, (len(Data.time), Nx, Ny), order='F')



In [5]:
data = xr.Dataset({'D_0': (['time','x', 'y'], D_0),
                   'D_prime': (['time','x', 'y'], D_prime),
                  },
                  coords={'x': xv[:,0],
                          'y': yv[0,:],
                          'z': -Data.z_r.data,
                          'time': Data.time.data},
                  attrs={'title': 'SUNTANS internally calculated energy budget'}) 


In [6]:

data.to_netcdf(param_result['resultfilepath']+param_result['result_filename'][:-3]+'_dissp.nc')

In [6]:
data["D0"]=(['time', 'x', 'y'],  D_0)
data["D_prime"]=(['time', 'x', 'y'],  D_prime)


In [7]:
del data

## _bt_flux

In [4]:
Fx_01 = np.reshape(Data.Fx_01_int.data, (len(Data.time), Nx, Ny), order='F')
Fx_02 = np.reshape(Data.Fx_02_int.data, (len(Data.time), Nx, Ny), order='F')
Fx_03 = np.reshape(Data.Fx_03_int.data, (len(Data.time), Nx, Ny), order='F')
Fx_04 = np.reshape(Data.Fx_04_int.data, (len(Data.time), Nx, Ny), order='F')

data_bt = xr.Dataset({'Fx_01_int': (['time','x', 'y'], Fx_01),
                  },
                  coords={'x': xv[:,0],
                          'y': yv[0,:],
                          'z': -Data.z_r.data,
                          'time': Data.time.data},
                  attrs={'title': 'SUNTANS internally calculated energy budget'}) 
data_bt["Fx_02_int"]=(['time', 'x', 'y'],  Fx_02)
data_bt["Fx_03_int"]=(['time', 'x', 'y'],  Fx_03)
data_bt["Fx_04_int"]=(['time', 'x', 'y'],  Fx_04)

del Fx_01, Fx_02, Fx_03, Fx_04

Fy_01 = np.reshape(Data.Fy_01_int.data, (len(Data.time), Nx, Ny), order='F')
Fy_02 = np.reshape(Data.Fy_02_int.data, (len(Data.time), Nx, Ny), order='F')
Fy_03 = np.reshape(Data.Fy_03_int.data, (len(Data.time), Nx, Ny), order='F')
Fy_04 = np.reshape(Data.Fy_04_int.data, (len(Data.time), Nx, Ny), order='F')

data_bt["Fy_01_int"]=(['time', 'x', 'y'],  Fy_01)
data_bt["Fy_02_int"]=(['time', 'x', 'y'],  Fy_02)
data_bt["Fy_03_int"]=(['time', 'x', 'y'],  Fy_03)
data_bt["Fy_04_int"]=(['time', 'x', 'y'],  Fy_04)

del Fy_01, Fy_02, Fy_03, Fy_04

data_bt.to_netcdf(param_result['resultfilepath']+param_result['result_filename'][:-3]+'_bt_flux.nc')

In [9]:
del data_bt

## _bc_flux

In [4]:
Fx_prime1 = np.reshape(Data.Fx_prime1_int.data, (len(Data.time), Nx, Ny), order='F')
Fx_prime2 = np.reshape(Data.Fx_prime2_int.data, (len(Data.time), Nx, Ny), order='F')
Fx_prime3 = np.reshape(Data.Fx_prime3_int.data, (len(Data.time), Nx, Ny), order='F')
Fx_prime4 = np.reshape(Data.Fx_prime4_int.data, (len(Data.time), Nx, Ny), order='F')
Fx_prime5 = np.reshape(Data.Fx_prime5_int.data, (len(Data.time), Nx, Ny), order='F')

data_bc = xr.Dataset({'Fx_prime1_int': (['time','x', 'y'], Fx_prime1),
                  },
                  coords={'x': xv[:,0],
                          'y': yv[0,:],
                          'z': -Data.z_r.data,
                          'time': Data.time.data},
                  attrs={'title': 'SUNTANS internally calculated energy budget'}) 
data_bc["Fx_prime2_int"]=(['time', 'x', 'y'],  Fx_prime2)
data_bc["Fx_prime3_int"]=(['time', 'x', 'y'],  Fx_prime3)
data_bc["Fx_prime4_int"]=(['time', 'x', 'y'],  Fx_prime4)
data_bc["Fx_prime5_int"]=(['time', 'x', 'y'],  Fx_prime5)

del Fx_prime1, Fx_prime2, Fx_prime3, Fx_prime4, Fx_prime5

Fy_prime1 = np.reshape(Data.Fy_prime1_int.data, (len(Data.time), Nx, Ny), order='F')
Fy_prime2 = np.reshape(Data.Fy_prime2_int.data, (len(Data.time), Nx, Ny), order='F')
Fy_prime3 = np.reshape(Data.Fy_prime3_int.data, (len(Data.time), Nx, Ny), order='F')
Fy_prime4 = np.reshape(Data.Fy_prime4_int.data, (len(Data.time), Nx, Ny), order='F')
Fy_prime5 = np.reshape(Data.Fy_prime5_int.data, (len(Data.time), Nx, Ny), order='F')

data_bc["Fy_prime1_int"]=(['time', 'x', 'y'],  Fy_prime1)
data_bc["Fy_prime2_int"]=(['time', 'x', 'y'],  Fy_prime2)
data_bc["Fy_prime3_int"]=(['time', 'x', 'y'],  Fy_prime3)
data_bc["Fy_prime4_int"]=(['time', 'x', 'y'],  Fy_prime4)
data_bc["Fy_prime5_int"]=(['time', 'x', 'y'],  Fy_prime5)

del Fy_prime1, Fy_prime2, Fy_prime3, Fy_prime4, Fy_prime5
del Data

data_bc.to_netcdf(param_result['resultfilepath']+param_result['result_filename'][:-3]+'_bc_flux.nc')

In [8]:
del data_bc

## _derived

In [4]:
data_bt_flux = xr.open_dataset(param_result['resultfilepath']+param_result['result_filename'][:-3]+'_bt_flux.nc')

div_F01 = data_bt_flux.Fx_01_int.differentiate("x") + data_bt_flux.Fy_01_int.differentiate("y")
div_F02 = data_bt_flux.Fx_02_int.differentiate("x") + data_bt_flux.Fy_02_int.differentiate("y")
div_F03 = data_bt_flux.Fx_03_int.differentiate("x") + data_bt_flux.Fy_03_int.differentiate("y")
div_F04 = data_bt_flux.Fx_04_int.differentiate("x") + data_bt_flux.Fy_04_int.differentiate("y")

del data_bt_flux

In [9]:
data_bt = xr.Dataset({'div_F01': (['time','x', 'y'], div_F01.data),
                  },
                  coords={'x': xv[:,0],
                          'y': yv[0,:],
                          'z': -Data.z_r.data,
                          'time': Data.time.data},
                  attrs={'title': 'SUNTANS internally calculated energy flux divergence'}) 

data_bt["div_F02"]=(['time', 'x', 'y'],  div_F02.data)
data_bt["div_F03"]=(['time', 'x', 'y'],  div_F03.data)
data_bt["div_F04"]=(['time', 'x', 'y'],  div_F04.data)

data_bt.to_netcdf(param_result['resultfilepath']+param_result['result_filename'][:-3]+'_bt.nc')

del div_F01, div_F02, div_F03, div_F04, data_bt

In [4]:
# read data
data_bc_flux = xr.open_dataset(param_result['resultfilepath']+param_result['result_filename'][:-3]+'_bc_flux.nc')

div_Fprime1 = data_bc_flux.Fx_prime1_int.differentiate("x") + data_bc_flux.Fy_prime1_int.differentiate("y")
div_Fprime2 = data_bc_flux.Fx_prime2_int.differentiate("x") + data_bc_flux.Fy_prime2_int.differentiate("y")
div_Fprime3 = data_bc_flux.Fx_prime3_int.differentiate("x") + data_bc_flux.Fy_prime3_int.differentiate("y")
div_Fprime4 = data_bc_flux.Fx_prime4_int.differentiate("x") + data_bc_flux.Fy_prime4_int.differentiate("y")
div_Fprime5 = data_bc_flux.Fx_prime5_int.differentiate("x") + data_bc_flux.Fy_prime5_int.differentiate("y")

del data_bc_flux

FileNotFoundError: [Errno 2] No such file or directory: '/Volumes/Tan_2024/SUNTANS/NLIW/iwave2024/dongsha/debug_withC_rogers_netcdfBdy1_EB_soliton_withsponge_focus_finey_summer/data_nof_1wave_largeAPE/results/sun_out_energy_derived_bc_flux.nc'

In [6]:
data_bc = xr.Dataset({'div_Fprime1': (['time','x', 'y'], div_Fprime1.data),
                  },
                  coords={'x': xv[:,0],
                          'y': yv[0,:],
                          'z': -Data.z_r.data,
                          'time': Data.time.data},
                  attrs={'title': 'SUNTANS internally calculated energy flux divergence'}) 

data_bc["div_Fprime2"]=(['time', 'x', 'y'],  div_Fprime2.data)
data_bc["div_Fprime3"]=(['time', 'x', 'y'],  div_Fprime3.data)
data_bc["div_Fprime4"]=(['time', 'x', 'y'],  div_Fprime4.data)
data_bc["div_Fprime5"]=(['time', 'x', 'y'],  div_Fprime5.data)

data_bc.to_netcdf(param_result['resultfilepath']+param_result['result_filename'][:-3]+'_bc.nc')

del div_Fprime1, div_Fprime2, div_Fprime3, div_Fprime4, div_Fprime5, data_bc

In [7]:
dissip0 = -data.dEp0 - data.dEk0 - data_bt.div_F01 - data_bt.div_F02 - data_bt.div_F03 - data_bt.div_F04 - data.C_int - data.D_0


AttributeError: 'Dataset' object has no attribute 'D_0'

In [3]:
data_bt = xr.open_dataset(param_result['resultfilepath']+param_result['result_filename'][:-3]+'_bt.nc')
data_bc = xr.open_dataset(param_result['resultfilepath']+param_result['result_filename'][:-3]+'_bc.nc')
data = xr.open_dataset(param_result['resultfilepath']+param_result['result_filename'])

# dissip0 = -data.dEp0 - data.dEk0 - data_bt.div_F01 - data_bt.div_F02 - data_bt.div_F03 - data_bt.div_F04 - data.C_int - data.D_0
# dissip_prime = -data.dEp_prime - data.dEk_prime - data_bc.div_Fprime1 - data_bc.div_Fprime2 - data_bc.div_Fprime3 - data_bc.div_Fprime4 - data_bc.div_Fprime5 + data.C_int - data.D_prime

# data["dissip0"]=(['time', 'x', 'y'],  dissip0.data)
# data["dissip_prime"]=(['time', 'x', 'y'],  dissip_prime.data)

In [11]:
dissip0 = -data.dEp0 - data.dEk0 - data_bt.div_F01 - data_bt.div_F02 - data_bt.div_F03 - data_bt.div_F04 - data.C_int - data.D0
# dissip_prime = -data.dEp_prime - data.dEk_prime - data_bc.div_Fprime1 - data_bc.div_Fprime2 - data_bc.div_Fprime3 - data_bc.div_Fprime4 - data_bc.div_Fprime5 + data.C_int - data.D_prime



In [10]:
data

<xarray.Dataset> Size: 30GB
Dimensions:    (time: 226, x: 1814, y: 1314, z: 100)
Coordinates:
  * x          (x) float64 15kB 40.01 212.6 385.0 ... 1.45e+05 1.451e+05
  * y          (y) float64 11kB 40.01 180.1 319.9 ... 1.05e+05 1.051e+05
  * z          (z) float64 800B -1.311 -3.973 -6.674 ... -576.0 -587.2 -598.7
  * time       (time) datetime64[ns] 2kB 2013-01-01T12:00:08 ... 2013-01-02T1...
Data variables:
    dEp0       (time, x, y) float64 4GB 0.0 0.0 0.0 0.0 ... 0.4441 0.4445 0.4445
    dEp_prime  (time, x, y) float64 4GB 1.298e-11 1.298e-11 ... 121.8 121.9
    dEk0       (time, x, y) float64 4GB ...
    dEk_prime  (time, x, y) float64 4GB ...
    C_int      (time, x, y) float64 4GB ...
    D0         (time, x, y) float64 4GB ...
    D_prime    (time, x, y) float64 4GB ...
    depth      (x, y) float64 19MB ...
Attributes:
    title:    SUNTANS internally calculated energy budget

In [15]:
data_ = xr.Dataset({'dissip0': (['time','x', 'y'], dissip0.data),
                  },
                  coords={'x': xv[:,0],
                          'y': yv[0,:],
                          'z': -Data.z_r.data,
                          'time': Data.time.data},
                  attrs={'title': 'SUNTANS internally calculated energy budget'}) 


In [16]:
data_["dissip0"]=(['time', 'x', 'y'],  dissip0.data)
data_["dissip_prime"]=(['time', 'x', 'y'],  dissip_prime.data)

In [8]:
del data_bt, data_bc, data, data_

In [18]:
data_.to_netcdf(param_result['resultfilepath']+param_result['result_filename'][:-3]+'_dissp.nc')


In [5]:
dissip0 = -data.dEp0 - data.dEk0 - data_bt.div_F01 - data_bt.div_F02 - data_bt.div_F03 - data_bt.div_F04 - data.C_int - data_.D_0


In [6]:
dissip_prime = -data.dEp_prime - data.dEk_prime - data_bc.div_Fprime1 - data_bc.div_Fprime2 - data_bc.div_Fprime3 - data_bc.div_Fprime4 - data_bc.div_Fprime5 + data.C_int - data_.D_prime


In [ ]:
data.to_netcdf(param_result['resultfilepath']+param_result['result_filename'])

del data_bt, data_bc, data

In [19]:
59734.31/3600

16.59286388888889

## _flux

In [3]:
data_bt_f = xr.open_dataset(param_result['resultfilepath']+param_result['result_filename'][:-3]+'_bt_flux.nc')
Fx_0 = data_bt_f.Fx_01_int.differentiate("time",datetime_unit="s")+data_bt_f.Fx_02_int.differentiate("time",datetime_unit="s")+data_bt_f.Fx_03_int.differentiate("time",datetime_unit="s")+data_bt_f.Fx_04_int.differentiate("time",datetime_unit="s")

In [4]:
data_f = xr.Dataset({'Fx_0': (['time','x', 'y'], Fx_0.data),
                  },
                  coords={'x': data_bt_f.x.data,
                          'y': data_bt_f.y.data,
                          'z': data_bt_f.z.data,
                          'time': data_bt_f.time.data},
                  attrs={'title': 'SUNTANS internally calculated energy flux'}) 

In [5]:
data_f.to_netcdf(param_result['resultfilepath']+param_result['result_filename'][:-3]+'_flux.nc')

In [4]:
Fy_0 = data_bt_f.Fy_01_int.differentiate("time",datetime_unit="s")+data_bt_f.Fy_02_int.differentiate("time",datetime_unit="s")+data_bt_f.Fy_03_int.differentiate("time",datetime_unit="s")+data_bt_f.Fy_04_int.differentiate("time",datetime_unit="s")

In [4]:
data_f = xr.open_dataset(param_result['resultfilepath']+param_result['result_filename'][:-3]+'_flux.nc')

In [6]:
data_f["Fy_0"]=(['time', 'x', 'y'],  Fy_0.data)

In [7]:
data_f.to_netcdf(param_result['resultfilepath']+param_result['result_filename'][:-3]+'_flux.nc')

In [3]:
data_bc_f = xr.open_dataset(param_result['resultfilepath']+param_result['result_filename'][:-3]+'_bc_flux_x.nc')
Fx_prime = data_bc_f.Fx_prime1_int.differentiate("time",datetime_unit="s")+data_bc_f.Fx_prime2_int.differentiate("time",datetime_unit="s")+data_bc_f.Fx_prime3_int.differentiate("time",datetime_unit="s")+data_bc_f.Fx_prime4_int.differentiate("time",datetime_unit="s")+data_bc_f.Fx_prime5_int.differentiate("time",datetime_unit="s")

In [5]:
data_f["Fx_prime"]=(['time', 'x', 'y'],  Fx_prime.data)

In [6]:
data_f.to_netcdf(param_result['resultfilepath']+param_result['result_filename'][:-3]+'_flux.nc')

In [3]:
data_bc_f = xr.open_dataset(param_result['resultfilepath']+param_result['result_filename'][:-3]+'_bc_flux_y.nc')
Fy_prime = data_bc_f.Fy_prime1_int.differentiate("time",datetime_unit="s")+data_bc_f.Fy_prime2_int.differentiate("time",datetime_unit="s")+data_bc_f.Fy_prime3_int.differentiate("time",datetime_unit="s")+data_bc_f.Fy_prime4_int.differentiate("time",datetime_unit="s")+data_bc_f.Fy_prime5_int.differentiate("time",datetime_unit="s")

In [5]:
data_f["Fy_prime"]=(['time', 'x', 'y'],  Fy_prime.data)

In [7]:
data_f.to_netcdf(param_result['resultfilepath']+param_result['result_filename'][:-3]+'_flux.nc')